In [ ]:
import pandas as pd
from scipy.io import mmread, mmwrite
from pathlib import Path
from ete3 import Tree
import numpy as np

# Set up project paths
project_root = Path('/workspaces/CellTreeBench')
data_dir = project_root / 'data' / 'celegans_small'
raw_dir = data_dir / 'raw' 
p0_dir = data_dir / 'P0'

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")
print(f"Raw data directory: {raw_dir}")
print(f"P0 directory: {p0_dir}")
print()

# Check if required directories exist
for dir_path in [raw_dir, p0_dir]:
    if not dir_path.exists():
        print(f"Warning: Directory does not exist: {dir_path}")
    else:
        print(f"✅ Directory exists: {dir_path}")


In [ ]:
# Load the curated P0 lineage tree
# This tree was built using the build_tree_celegans_small notebooks

print("Loading curated P0 lineage tree...")
tree_file = p0_dir / "tree_df-P0.csv"

if not tree_file.exists():
    print(f"❌ Error: P0 tree file not found: {tree_file}")
    print("Please run the build_tree_celegans_small notebooks first!")
    raise FileNotFoundError(f"P0 tree file not found: {tree_file}")

tree_df = pd.read_csv(tree_file)
print(f"✅ Loaded P0 tree with {len(tree_df)} nodes")

# Show tree statistics
leaf_nodes = tree_df[~tree_df['Lineage'].isin(tree_df['Parent'].dropna())]
print(f"Tree statistics:")
print(f"  Total nodes: {len(tree_df)}")
print(f"  Leaf nodes: {len(leaf_nodes)}")
print(f"  Internal nodes: {len(tree_df) - len(leaf_nodes)}")

tree_df.head()


In [ ]:
# Load raw expression data and check what we have
print("Loading raw expression data...")

# Check what expression files are available
print("Available expression files:")
for file_pattern in ['*.mm', 'exprs*.csv', '*expression*']:
    files = list(raw_dir.glob(file_pattern))
    for f in files:
        print(f"  {f.name} ({f.stat().st_size / (1024**3):.2f} GB)")

# Load cell barcodes
barcodes_file = raw_dir / "cell_barcodes.csv"
if barcodes_file.exists():
    barcodes_list = pd.read_csv(barcodes_file, header=None).values.flatten()
    print(f"✅ Loaded {len(barcodes_list)} cell barcodes")
else:
    print(f"❌ Cell barcodes file not found: {barcodes_file}")

# Load gene names
genes_file = raw_dir / "genes.csv"
if genes_file.exists():
    genes_list = pd.read_csv(genes_file, header=None).values.flatten()
    print(f"✅ Loaded {len(genes_list)} genes")
else:
    print(f"❌ Genes file not found: {genes_file}")

# Try to load expression data from available sources
expression_loaded = False

# Try normalized expression first
exprs_normalized_file = raw_dir / "exprs_normalized.mm"
if exprs_normalized_file.exists():
    print(f"Loading normalized expression matrix: {exprs_normalized_file}")
    sparse_matrix = mmread(exprs_normalized_file)
    exprs_df = pd.DataFrame(sparse_matrix.toarray())
    exprs_df.index = barcodes_list
    exprs_df.columns = genes_list
    expression_loaded = True
    print(f"✅ Loaded normalized expression data: {exprs_df.shape}")
    
# Fallback to regular expression data
elif (raw_dir / "exprs.mm").exists():
    exprs_file = raw_dir / "exprs.mm"
    print(f"Loading expression matrix: {exprs_file}")
    sparse_matrix = mmread(exprs_file)
    exprs_df = pd.DataFrame(sparse_matrix.toarray())
    exprs_df.index = barcodes_list
    exprs_df.columns = genes_list
    expression_loaded = True
    print(f"✅ Loaded expression data: {exprs_df.shape}")

if not expression_loaded:
    print("❌ No expression data files found!")
    raise FileNotFoundError("Expression data not found")

# Show summary
print(f"\nExpression data summary:")
print(f"  Shape: {exprs_df.shape}")
print(f"  Non-zero values: {(exprs_df > 0).sum().sum():,}")
print(f"  Sparsity: {((exprs_df == 0).sum().sum() / exprs_df.size * 100):.1f}%")

exprs_df.head()


In [ ]:
# Load cell metadata and map cells to lineages
print("Loading cell metadata...")
metadata_file = raw_dir / "metadata.csv"

if not metadata_file.exists():
    print(f"❌ Metadata file not found: {metadata_file}")
    raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

# Load metadata with proper settings
metadata_df = pd.read_csv(metadata_file, low_memory=False)
print(f"✅ Loaded metadata for {len(metadata_df)} cells")

# Show metadata columns
print(f"\nMetadata columns ({len(metadata_df.columns)}):")
for i, col in enumerate(metadata_df.columns):
    print(f"  {i+1:2d}. {col}")

# Get available lineages in the data
if 'lineage_packer' in metadata_df.columns:
    available_lineages = metadata_df['lineage_packer'].dropna().unique()
    print(f"\nAvailable lineages in metadata: {len(available_lineages)}")
    print("First 20 lineages:", available_lineages[:20])
else:
    print("❌ 'lineage_packer' column not found in metadata")

# Show metadata summary
print(f"\nMetadata summary:")
print(f"  Total cells: {len(metadata_df)}")
print(f"  Cells with lineage info: {metadata_df['lineage_packer'].notna().sum()}")
print(f"  Unique lineages: {metadata_df['lineage_packer'].nunique()}")

metadata_df.head()


In [ ]:
# Map cells to tree lineages and filter data
print("Mapping cells to tree lineages...")

# Get lineages that are present in our curated tree
tree_lineages = set(tree_df['Lineage'].values)
print(f"Tree contains {len(tree_lineages)} lineages")

# Get lineages available in metadata  
if 'lineage_packer' in metadata_df.columns:
    metadata_lineages = set(metadata_df['lineage_packer'].dropna().values)
    print(f"Metadata contains {len(metadata_lineages)} lineages")
    
    # Find intersection
    common_lineages = tree_lineages.intersection(metadata_lineages)
    print(f"Common lineages: {len(common_lineages)}")
    
    # Show some examples
    print(f"Examples of common lineages: {list(common_lineages)[:10]}")
    
    # Find lineages in tree but not in metadata
    tree_only = tree_lineages - metadata_lineages
    if tree_only:
        print(f"Lineages in tree but not in metadata ({len(tree_only)}): {list(tree_only)[:10]}")
    
    # Find lineages in metadata but not in tree
    metadata_only = metadata_lineages - tree_lineages
    if metadata_only:
        print(f"Lineages in metadata but not in tree ({len(metadata_only)}): {list(metadata_only)[:10]}")
        
else:
    print("❌ Cannot map lineages - 'lineage_packer' column not found")
    common_lineages = set()


In [ ]:
# Filter data to include only cells with lineages in our tree
print("Filtering data based on tree lineages...")

if common_lineages:
    # Filter metadata to include only cells with lineages in our tree
    filtered_metadata = metadata_df[metadata_df['lineage_packer'].isin(common_lineages)].copy()
    print(f"Filtered metadata: {len(filtered_metadata)} cells (from {len(metadata_df)})")
    
    # Filter expression data to match filtered metadata
    common_cells = set(filtered_metadata['cell'].values).intersection(set(exprs_df.index))
    print(f"Common cells in both metadata and expression data: {len(common_cells)}")
    
    if common_cells:
        # Filter both datasets to common cells
        filtered_metadata = filtered_metadata[filtered_metadata['cell'].isin(common_cells)]
        filtered_exprs = exprs_df.loc[list(common_cells)]
        
        print(f"Final filtered data:")
        print(f"  Cells: {len(filtered_metadata)}")
        print(f"  Expression shape: {filtered_exprs.shape}")
        print(f"  Lineages represented: {filtered_metadata['lineage_packer'].nunique()}")
        
        # Add lineage_packer as 'leaf' column for dataset compatibility
        filtered_metadata['leaf'] = filtered_metadata['lineage_packer']
        
        # Update cell counts in tree
        print("\nUpdating cell counts in tree...")
        tree_df_updated = tree_df.copy()
        
        for lineage in tree_df_updated['Lineage']:
            n_cells = (filtered_metadata['lineage_packer'] == lineage).sum()
            tree_df_updated.loc[tree_df_updated['Lineage'] == lineage, 'n_cells'] = n_cells
        
        # Show updated cell counts
        leaf_nodes_updated = tree_df_updated[~tree_df_updated['Lineage'].isin(tree_df_updated['Parent'].dropna())]
        total_cells_in_leaves = leaf_nodes_updated['n_cells'].sum()
        print(f"Total cells in leaf nodes: {total_cells_in_leaves}")
        print(f"Leaf nodes with cells: {(leaf_nodes_updated['n_cells'] > 0).sum()}")
        
    else:
        print("❌ No common cells found between metadata and expression data")
        filtered_metadata = pd.DataFrame()
        filtered_exprs = pd.DataFrame()
        tree_df_updated = tree_df.copy()
        
else:
    print("❌ No common lineages found - cannot proceed with filtering")
    filtered_metadata = pd.DataFrame()
    filtered_exprs = pd.DataFrame()
    tree_df_updated = tree_df.copy()

# Show summary
print(f"\nData filtering summary:")
print(f"  Original cells: {len(metadata_df)}")
print(f"  Filtered cells: {len(filtered_metadata)}")
print(f"  Expression data shape: {filtered_exprs.shape if not filtered_exprs.empty else 'N/A'}")

tree_df_updated.head()


In [ ]:
# Save processed data in formats compatible with CElegansDatasetBase
print("Saving processed data for CElegansDatasetBase...")

if not filtered_metadata.empty and not filtered_exprs.empty:
    # Ensure output directories exist
    raw_dir.mkdir(parents=True, exist_ok=True)
    p0_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save updated tree with cell counts (overwrite the existing one)
    tree_output_file = p0_dir / "tree_df-P0.csv"
    tree_df_updated.to_csv(tree_output_file, index=False)
    print(f"✅ Saved updated tree: {tree_output_file}")
    
    # 2. Save filtered metadata 
    metadata_output_file = raw_dir / "metadata.csv"
    filtered_metadata.to_csv(metadata_output_file, index=False)
    print(f"✅ Saved filtered metadata: {metadata_output_file} ({len(filtered_metadata)} cells)")
    
    # 3. Save expression data in Matrix Market format (sparse, efficient)
    # Reorder expression data to match metadata order
    ordered_exprs = filtered_exprs.loc[filtered_metadata['cell']]
    
    # Convert to sparse matrix and save
    from scipy.sparse import csr_matrix
    sparse_exprs = csr_matrix(ordered_exprs.values)
    exprs_output_file = raw_dir / "exprs.mm"
    mmwrite(exprs_output_file, sparse_exprs)
    print(f"✅ Saved expression matrix: {exprs_output_file} {sparse_exprs.shape}")
    
    # 4. Save cell barcodes (in same order as expression matrix)
    barcodes_output_file = raw_dir / "cell_barcodes.csv"
    pd.DataFrame(ordered_exprs.index).to_csv(barcodes_output_file, index=False, header=False)
    print(f"✅ Saved cell barcodes: {barcodes_output_file}")
    
    # 5. Save gene names (columns of expression matrix)
    genes_output_file = raw_dir / "genes.csv"
    pd.DataFrame(ordered_exprs.columns).to_csv(genes_output_file, index=False, header=False)
    print(f"✅ Saved gene names: {genes_output_file}")
    
    print(f"\n{'='*60}")
    print("PREPROCESSING COMPLETE!")
    print(f"{'='*60}")
    print("Generated files for CElegansDatasetBase:")
    print(f"📁 {p0_dir}/")
    print(f"   └── tree_df-P0.csv                 # Tree structure with cell counts")
    print(f"📁 {raw_dir}/")
    print(f"   ├── metadata.csv                   # Cell metadata with lineage mapping")
    print(f"   ├── exprs.mm                       # Expression matrix (sparse)")
    print(f"   ├── cell_barcodes.csv              # Cell identifiers")
    print(f"   └── genes.csv                      # Gene names")
    print()
    print("Dataset statistics:")
    print(f"  📊 Cells: {len(filtered_metadata):,}")
    print(f"  📊 Genes: {ordered_exprs.shape[1]:,}")
    print(f"  📊 Lineages: {filtered_metadata['lineage_packer'].nunique()}")
    print(f"  📊 Leaf nodes with cells: {(tree_df_updated['n_cells'] > 0).sum()}")
    print()
    print("Ready for CElegansDatasetBase(dataset_name='celegans_small')!")
    
else:
    print("❌ Cannot save data - no filtered data available")
    print("Please check the data filtering step above.")

# Show final data sample
if not filtered_metadata.empty:
    print(f"\nFinal metadata sample:")
    filtered_metadata.head()


In [ ]:
# Optional: Verify the generated dataset can be loaded by CElegansDatasetBase
try:
    print("Testing dataset loading...")
    print("Note: This requires CElegansDatasetBase to be importable")
    
    # Try to import and test (this might fail if not in the right environment)
    try:
        import sys
        sys.path.append(str(project_root / 'src'))
        from celltreebench.datasets.celegans_dataset_base import CElegansDatasetBase
        
        # Test dataset loading
        dataset = CElegansDatasetBase(
            dataset_name="celegans_small",
            lineage_name="P0",
            data_dir=str(data_dir.parent)
        )
        
        print(f"✅ Dataset loaded successfully!")
        print(f"   Number of leaves: {dataset.n_leaves}")
        print(f"   Tree file: {dataset.topology_tree}")
        print(f"   Metadata shape: {dataset.metadata_df.shape}")
        print(f"   Expression shape: {dataset.exprs_df.shape}")
        
    except ImportError as e:
        print(f"⚠️  Cannot import CElegansDatasetBase: {e}")
        print("This is normal if running outside the full project environment")
    except Exception as e:
        print(f"⚠️  Error testing dataset: {e}")
        print("Please check if the generated files are correctly formatted")
        
except Exception as e:
    print(f"❌ Error in dataset verification: {e}")

print("\n" + "="*60)
print("PREPROCESSING WORKFLOW COMPLETE!")
print("="*60)
